# Lung Nematic

Nematic director fields and **candidate topological defects** (±1/2) in lung
histology (H&E), with three orientation sources and statistical controls:

- **nuclear** field (from nuclear long axes),
- **collagen** field (structure tensor of the eosin channel),
- **fused** field (nuclei where cells are dense, collagen where fibers are),
- **permutation null model** (are there more/fewer defects than by chance?),
- **colocalization test** (do defects sit in structured regions?).

Run the cells top to bottom. The code is hidden — you only see titles and
controls. Adjust parameters and re-run **Run analysis** as needed.

In [ ]:
#@title 🔧 Setup — run once { display-mode: "form" }
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "git+https://github.com/Danpc11/lung-nematic.git"], check=True)
import lung_nematic
print("Ready. lung_nematic", lung_nematic.__version__)

In [ ]:
#@title 📁 Load images — upload or from Drive { display-mode: "form" }
image_source = "Upload from computer" #@param ["Upload from computer", "Google Drive folder"]
#@markdown Drive path, e.g. `MyDrive/histology`.
drive_folder = "MyDrive/lung_histology" #@param {type:"string"}

import os, shutil, glob
IMAGES_DIR = "/content/images"
EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")
if os.path.isdir(IMAGES_DIR):
    shutil.rmtree(IMAGES_DIR)
os.makedirs(IMAGES_DIR, exist_ok=True)

if image_source == "Upload from computer":
    from google.colab import files
    for name in files.upload():
        shutil.move(name, os.path.join(IMAGES_DIR, os.path.basename(name)))
else:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    src = drive_folder if os.path.isabs(drive_folder) else os.path.join("/content/drive", drive_folder)
    if not os.path.isdir(src):
        raise FileNotFoundError(f"Drive folder not found: {src}")
    found = [p for p in sorted(glob.glob(os.path.join(src, "*"))) if p.lower().endswith(EXTS)]
    if not found:
        raise FileNotFoundError(f"No {EXTS} images in {src}")
    for p in found:
        shutil.copy(p, os.path.join(IMAGES_DIR, os.path.basename(p)))

names = sorted(os.listdir(IMAGES_DIR))
print(f"{len(names)} image(s) ready:")
for n in names: print("  ", n)

In [ ]:
#@title ⚙️ Analysis parameters { display-mode: "form", run: "auto" }
#@markdown Coarse-graining scales for the director field (px, comma-separated).
sigmas_px = "40, 55, 70, 85" #@param {type:"string"}
defect_grid_step_px = 24 #@param {type:"slider", min:8, max:64, step:4}
min_edge_distance_px = 30 #@param {type:"slider", min:0, max:120, step:5}
density_quantile = 0.45 #@param {type:"slider", min:0, max:0.9, step:0.05}
min_scales_for_persistence = 2 #@param {type:"slider", min:1, max:4, step:1}
defect_cluster_radius_px = 70 #@param {type:"slider", min:20, max:150, step:5}
#@markdown Image scale (µm/pixel). Leave 0 if unknown.
microns_per_pixel = 0.0 #@param {type:"number"}

from lung_nematic.config import AnalysisConfig
try:
    _sigmas = tuple(float(s) for s in sigmas_px.split(",") if s.strip())
    config = AnalysisConfig(
        sigmas_px=_sigmas,
        defect_grid_step_px=int(defect_grid_step_px),
        min_edge_distance_px=int(min_edge_distance_px),
        density_quantile=float(density_quantile),
        min_scales_for_persistence=min(int(min_scales_for_persistence), len(_sigmas)),
        defect_cluster_radius_px=float(defect_cluster_radius_px),
        default_microns_per_pixel=(microns_per_pixel or None),
    )
    config.validate()
    print("Analysis parameters set.")
except Exception as error:
    print("Adjust parameters:", error)

In [ ]:
#@title 🧭 What to run { display-mode: "form", run: "auto" }
#@markdown Orientation source for the director field.
field_type = "collagen" #@param ["nuclear", "collagen", "fused"]
#@markdown Permutation null model (nuclear and collagen only; skipped for fused).
run_null = True #@param {type:"boolean"}
n_permutations = 199 #@param {type:"slider", min:19, max:999, step:10}
downsample = 3 #@param {type:"slider", min:1, max:6, step:1}
#@markdown Nuclear null only: "shuffle" keeps the angle marginal, "uniform" does not.
null_mode = "shuffle" #@param ["shuffle", "uniform"]
null_seed = 0 #@param {type:"integer"}
#@markdown Colocalization test (local order at defects vs random tissue).
run_coloc = True #@param {type:"boolean"}
n_bootstrap = 2000 #@param {type:"slider", min:200, max:5000, step:100}
print(f"Field: {field_type} | null: {run_null} | colocalization: {run_coloc}")

In [ ]:
#@title ▶️ Run analysis { display-mode: "form" }
import os, glob, warnings, shutil
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

from lung_nematic.io_utils import read_rgb
from lung_nematic.preprocessing import make_tissue_mask
from lung_nematic.segmentation import segment_nuclei, select_oriented_nuclei
from lung_nematic.nematic import get_density_threshold
from lung_nematic.defects import detect_multiscale_defects
from lung_nematic.collagen_field import detect_multiscale_collagen_defects
from lung_nematic.fused_field import detect_multiscale_fused_defects
from lung_nematic.null_model import run_null_model, run_collagen_null_model, save_null_histogram
from lung_nematic.colocalization import run_colocalization

RESULTS_DIR = "/content/results"
if os.path.isdir(RESULTS_DIR): shutil.rmtree(RESULTS_DIR)
os.makedirs(RESULTS_DIR, exist_ok=True)

def save_overlay(rgb, mask, field, defects, path, title):
    fig, ax = plt.subplots(figsize=(13, 10)); ax.imshow(rgb); ax.axis("off"); ax.set_title(title)
    step = config.field_grid_step_px; H, W = mask.shape
    cut = get_density_threshold(field["density"], mask, config.density_quantile)
    for y in range(step//2, H, step):
        for x in range(step//2, W, step):
            if not (mask[y,x] and field["density"][y,x] > cut
                    and field["order"][y,x] >= config.min_local_order_for_display):
                continue
            L = 6 + 24*field["order"][y,x]
            dx = L*np.cos(field["theta"][y,x]); dy = L*np.sin(field["theta"][y,x])
            ax.plot([x-dx,x+dx],[y-dy,y+dy], color="#0072B2", linewidth=1.0)
    if not defects.empty:
        p = defects[defects.charge==0.5]; n = defects[defects.charge==-0.5]
        ax.scatter(p.x_px, p.y_px, marker="+", s=180, linewidths=2.5, color="#111", label="+1/2")
        ax.scatter(n.x_px, n.y_px, marker="x", s=150, linewidths=2.5, color="#D55E00", label="-1/2")
        ax.legend(loc="lower left")
    fig.tight_layout(); fig.savefig(path, dpi=140, bbox_inches="tight"); plt.close(fig)

def save_coloc_hist(result, path, title):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    s = np.asarray(result["null_scores"])
    if s.size: ax.hist(s, bins=30, alpha=0.8)
    ax.axvline(result["observed_mean_score"], color="crimson", lw=2.5,
               label=f"observed = {result['observed_mean_score']:.3f}")
    ax.axvline(result["null_mean"], color="black", ls="--", lw=1.5,
               label=f"null = {result['null_mean']:.3f}")
    ax.set_xlabel("mean local order at defect locations"); ax.set_ylabel("bootstrap draws")
    ax.set_title(f"{title}\n{result['direction']} | p(two-sided)={result['p_two_sided']:.3f} | z={result['z_score']:.2f}")
    ax.legend(); fig.tight_layout(); fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)

images = sorted(glob.glob("/content/images/*"))
if not images:
    raise RuntimeError("No images loaded. Run the load-images cell first.")
rep = float(config.sigmas_px[len(config.sigmas_px)//2])
rows = []

for path in tqdm(images, desc="Analyzing"):
    image_id = os.path.splitext(os.path.basename(path))[0]
    out = os.path.join(RESULTS_DIR, image_id); os.makedirs(out, exist_ok=True)
    rgb = read_rgb(path); mask, hed = make_tissue_mask(rgb); eosin = hed[:, :, 1]
    oriented = None
    if field_type in ("nuclear", "fused"):
        _, nuclei = segment_nuclei(mask, hed, config)
        oriented = select_oriented_nuclei(nuclei, config)

    if field_type == "nuclear":
        defects, fields, _ = detect_multiscale_defects(oriented, mask, config)
    elif field_type == "collagen":
        defects, fields, _ = detect_multiscale_collagen_defects(eosin, mask, config)
    else:
        defects, fields, _ = detect_multiscale_fused_defects(oriented, eosin, mask, config)

    save_overlay(rgb, mask, fields[rep], defects,
                 os.path.join(out, f"{image_id}_{field_type}_overlay.png"),
                 f"{image_id} | {field_type} | {len(defects)} defects")

    row = {"image_id": image_id, "field": field_type, "n_defects": len(defects),
           "n_plus_half": int((defects.charge==0.5).sum()) if not defects.empty else 0,
           "n_minus_half": int((defects.charge==-0.5).sum()) if not defects.empty else 0}

    if run_null and field_type in ("nuclear", "collagen"):
        if field_type == "nuclear":
            nres = run_null_model(oriented, mask, config, n_permutations=int(n_permutations),
                                  downsample=int(downsample), mode=null_mode, seed=int(null_seed))
        else:
            nres = run_collagen_null_model(eosin, mask, config, n_permutations=int(n_permutations),
                                           downsample=int(downsample), seed=int(null_seed))
        save_null_histogram(nres, os.path.join(out, f"{image_id}_{field_type}_null_hist.png"), title=f"{image_id} ({field_type})")
        for k in ["null_mean","null_std","z_score","p_two_sided","direction"]:
            row[f"null_{k}"] = nres[k]

    if run_coloc:
        cres = run_colocalization(defects, fields[rep], mask, config, n_bootstrap=int(n_bootstrap), seed=int(null_seed))
        save_coloc_hist(cres, os.path.join(out, f"{image_id}_{field_type}_coloc_hist.png"), title=f"{image_id} ({field_type})")
        for k in ["observed_mean_score","null_mean","z_score","p_two_sided","direction"]:
            row[f"coloc_{k}"] = cres[k]
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_csv(os.path.join(RESULTS_DIR, "summary.csv"), index=False)
print("Done.")
summary_df

In [ ]:
#@title 📊 Show results { display-mode: "form" }
import glob, os
import pandas as pd
from IPython.display import Image as IPyImage, display
for overlay in sorted(glob.glob("/content/results/*/*_overlay.png")):
    folder = os.path.dirname(overlay); image_id = os.path.basename(folder)
    print("=" * 60); print(image_id)
    display(IPyImage(overlay, width=880))
    for extra in sorted(glob.glob(os.path.join(folder, "*_null_hist.png")) +
                        glob.glob(os.path.join(folder, "*_coloc_hist.png"))):
        display(IPyImage(extra, width=520))
sp = "/content/results/summary.csv"
if os.path.exists(sp):
    print("Summary:"); display(pd.read_csv(sp))

In [ ]:
#@title ⬇️ Download results (zip) { display-mode: "form" }
import shutil
from google.colab import files
archive = shutil.make_archive("/content/lung_nematic_results", "zip", "/content/results")
files.download(archive)